# 01 — Smokes, Friends, Cancer

The canonical LTN example. We have a small social network of people.  
We know who smokes, who is friends with whom, and who has cancer (partially).  
We encode domain knowledge as **first-order logic axioms** and train the model to **satisfy** them.

**What you will see:**
- How symbols (people, predicates) are *grounded* as tensors / neural networks
- How fuzzy connectives replace Boolean logic
- How `Forall` and `Exists` are implemented as **fold (reduce) operations** over truth-value tensors
- The LTN training loop: maximise satisfaction ↔ minimise `1 - sat`

**Source:** `examples/smokes_friends_cancer/smokes_friends_cancer.py`

## 0. Imports

In [1]:
import sys
sys.path.insert(0, '..')   # so that `import ltn` finds the local package

import numpy as np
import tensorflow as tf
import ltn

## 1. The domain: people and facts

We have 14 people split into two groups:
- **g1** = `{a,b,c,d,e,f,g,h}` — cancer labels are *known*
- **g2** = `{i,j,k,l,m,n}` — cancer labels are *unknown* (semi-supervised)

Each person is a **trainable constant**: a vector in ℝ¹⁰ learned during training.  
Think of it as a learned embedding for that individual.

In [2]:
EMBEDDING_SIZE = 10

# ltn.Constant(value, trainable=True) wraps the vector in a tf.Variable so gradients flow through it
g1 = {l: ltn.Constant(np.random.uniform(0., 1., size=EMBEDDING_SIZE), trainable=True) for l in 'abcdefgh'}
g2 = {l: ltn.Constant(np.random.uniform(0., 1., size=EMBEDDING_SIZE), trainable=True) for l in 'ijklmn'}
g  = {**g1, **g2}

# Known facts from the dataset
friends = [('a','b'),('a','e'),('a','f'),('a','g'),('b','c'),('c','d'),
           ('e','f'),('g','h'),('i','j'),('j','m'),('k','l'),('m','n')]
smokes  = ['a','e','f','g','j','n']
cancer  = ['a','e']

print(f"People: {list(g.keys())}")
print(f"Smokers: {smokes}")
print(f"Cancer (known): {cancer}")

People: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n']
Smokers: ['a', 'e', 'f', 'g', 'j', 'n']
Cancer (known): ['a', 'e']


---
### ❓ Q1 — Are g1 and g2 symbols? Do I always have to define them this way?

**Yes, g1 and g2 contain *constants* — the LTN term for named individuals (symbols) in the domain.**

In first-order logic, constants are the atomic objects your theory talks about: `anna`, `bob`, `cancer_drug_A`, pixel `(3,7)`, etc.  
In LTN each constant is *grounded* as a numeric vector — that is the bridge between symbols and tensors.

**Do you have to define them this way?** No, the dict+loop pattern is just convenience. The only requirement is that each individual is an `ltn.Constant`. You could equally write:

```python
anna = ltn.Constant([0.3, 0.7, ...], trainable=True)
bob  = ltn.Constant([0.1, 0.5, ...], trainable=True)
```

The two-group split (g1/g2) is **domain-specific**: we split because cancer supervision is only available for g1. In a problem where all labels are known (or none are), a single dict would be fine.

Also: constants don't have to be trainable. If you already know the representation of each individual (e.g. a fixed feature vector from a sensor), set `trainable=False`.

Smokes  = ltn.Predicate.MLP(([EMBEDDING_SIZE],),                  hidden_layer_sizes=(16,16))
Friends = ltn.Predicate.MLP(([EMBEDDING_SIZE], [EMBEDDING_SIZE]), hidden_layer_sizes=(16,16))
Cancer  = ltn.Predicate.MLP(([EMBEDDING_SIZE],),                  hidden_layer_sizes=(16,16))

# Quick sanity check: evaluate Smokes on person 'a'
print("Smokes('a') before training:", Smokes(g['a']).tensor.numpy())

---
### ❓ Q3 — Why is the embedding size 10? Is it arbitrary?

**Yes, it is a hyperparameter** — there is no magic in the number 10.

The embedding size determines how much *capacity* each individual has to represent itself. The constraints are:

- **Too small** (e.g. 2): the model may not have enough degrees of freedom to simultaneously satisfy all axioms — individuals that need to be distinguished may end up at the same point.
- **Too large** (e.g. 1000): the model can trivially satisfy everything on the training individuals but generalises poorly; also slower.

For a toy problem with 14 people and a handful of predicates, anything from 4 to 20 works fine. In the original LTN papers, values like 2, 10, or 64 appear depending on domain complexity.

**Rule of thumb:** start small and increase if the sat level plateaus low. The embedding size should grow with the number of individuals and the complexity of the predicates.

## 2. Grounding predicates as MLPs

In LTN, a **predicate** $P(x)$ is a function that maps an embedding vector to $[0,1]$ (a fuzzy truth value).  
We ground it as a small MLP with a sigmoid output.

- `Smokes(person)` → scalar in [0,1]
- `Friends(person1, person2)` → scalar in [0,1]  
- `Cancer(person)` → scalar in [0,1]

In [9]:
Smokes  = ltn.Predicate.MLP(([EMBEDDING_SIZE],),                    hidden_layer_sizes=(16,16))
Friends = ltn.Predicate.MLP(([EMBEDDING_SIZE, EMBEDDING_SIZE]),    hidden_layer_sizes=(16,16))
Cancer  = ltn.Predicate.MLP(([EMBEDDING_SIZE], ),                   hidden_layer_sizes=(16,16))

# Quick sanity check: evaluate Smokes on person 'a'
print("Smokes('a') before training:", Smokes(g['a']).tensor.numpy())

ValueError: Cannot convert '10' to a shape.

## 3. Fuzzy connectives

Classical Boolean logic is replaced with continuous, differentiable alternatives:

| Symbol | Fuzzy implementation | Formula |
|--------|---------------------|--------|
| ¬x | `Not_Std` | 1 − x |
| x ∧ y | `And_Prod` | x · y |
| x ∨ y | `Or_ProbSum` | x + y − x·y |
| x ⟹ y | `Implies_Reichenbach` | 1 − x + x·y |

All operators are element-wise on tensors — broadcasting handles multiple variable axes automatically.

In [ ]:
Not     = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And     = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or      = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())

---
### ❓ Q6 — You use Implies_Reichenbach. Are there other algebras? What is their impact?

**Yes — fuzzy logic has multiple families of connectives, each forming a consistent algebra.**

The choice comes from the theory of *t-norms* (generalisations of AND to [0,1]). Each t-norm induces a compatible implication:

| Name | Implies formula | And formula | Property |
|------|----------------|-------------|----------|
| **Łukasiewicz** | min(1, 1−x+y) | max(0, x+y−1) | Bounded; gradient vanishes when x+y < 1 |
| **Gödel** | 1 if x≤y else y | min(x,y) | Piecewise; zero gradient almost everywhere (bad for learning) |
| **Reichenbach** (product) | 1−x+x·y | x·y | Smooth everywhere; recommended for gradient-based training |
| **Goguen** | 1 if x≤y else y/x | x·y | Smooth but can produce large gradients near x=0 |

**Practical impact:**
- **Gödel** is the classical min/max logic. It has almost-zero gradients everywhere (non-differentiable at the kink), so it is **not suitable for learning**.
- **Łukasiewicz** clips to 0 when premises are jointly weak, which can kill gradients early in training.
- **Reichenbach** (product t-norm) is smooth and has gradients everywhere — that is why it is the default for LTN learning.
- The choice affects **how strictly implications are enforced** and **how gradients flow**. For inference-only use, Gödel may be preferred (exact min/max semantics). For learning, Reichenbach is usually the best starting point.

All options are available in `ltn/fuzzy_ops.py`: `Implies_Luk`, `Implies_Godel`, `Implies_Reichenbach`, `Implies_Goguen`.

## 4. Quantifiers = FOLD over truth values  ← the key idea

This is where the **fold** happens.

In classical logic, `∀x P(x)` is true iff *every* individual satisfies P.  
In LTN, `Forall(x, P(x))` **reduces** (folds) the tensor of truth values `[P(x₁), P(x₂), ..., P(xₙ)]`  
into a single scalar using a **generalised mean** called *pMeanError*:

$$\text{Forall}(x, P(x)) = 1 - \left(\frac{1}{n}\sum_{i=1}^{n}(1 - P(x_i))^p\right)^{1/p}$$

- When p→∞ this approaches `min` (strict: all must be true)
- When p=1 this is the arithmetic mean
- In practice p=2 gives a smooth, differentiable approximation of `min`

`Exists` uses *pMean* (approximation of `max`):
$$\text{Exists}(x, P(x)) = \left(\frac{1}{n}\sum_{i=1}^{n} P(x_i)^p\right)^{1/p}$$

Both are implemented in `ltn/fuzzy_ops.py` as `Aggreg_pMeanError` and `Aggreg_pMean`.

---
### ❓ Q4 — What is a fold?

**A fold (also called reduce) is a fundamental operation in functional programming: it collapses a collection of values into a single value by repeatedly applying a binary operation.**

Example with Python's `functools.reduce`:

```python
from functools import reduce
values = [0.9, 0.7, 0.4, 0.8]  # truth values for P(x1), P(x2), P(x3), P(x4)

# fold with min → strict Forall
reduce(min, values)   # 0.4

# fold with + then /n → arithmetic mean
sum(values) / len(values)   # 0.7
```

In NumPy/TensorFlow this is `np.min`, `tf.reduce_mean`, `tf.reduce_sum` — all are folds.

**In LTN:** when you write `Forall(x, P(x))`, the framework:
1. Evaluates `P` on every individual at once → tensor `[P(x₁), ..., P(xₙ)]`
2. **Folds** that tensor to a scalar using `Aggreg_pMeanError`

The fold is what turns a formula with a free variable into a closed formula (a single truth value).  
This is the direct analogue of the classical semantics: `∀x P(x) ≡ P(x₁) ∧ P(x₂) ∧ ... ∧ P(xₙ)` — a big AND is itself a fold over conjunction.

---
### ❓ Q5 — Where do the values p=2 (Forall) and p=6 (Exists) come from? Is Forall always pMeanError and Exists always pMean?

**The p values are hyperparameters. The values 2 and 6 come from the original LTN paper as reasonable defaults, not from a theorem.**

Understanding what p does:

**For `Forall` (pMeanError):** measures the *average error* (how far from 1 each truth value is).  
Higher p → more weight on the worst-case violator → approaches `min`.

| p | Forall behaviour |
|---|------------------|
| 1 | arithmetic mean of errors — lenient, outliers don't dominate |
| 2 | quadratic mean — moderate, balances average and worst case |
| ∞ | exact min — fails if even one individual violates the formula |

p=2 is chosen because it is strict enough to learn meaningful universals but smooth enough for gradients to flow.

**For `Exists` (pMean):** measures the *average truth*.  
Higher p → more weight on the best individual → approaches `max`.

p=6 is higher (more selective) so that `Exists` only fires when there is a genuinely good witness, not just a mild average. Note in the code `p_exists` is even used as a curriculum: start at p=1 (easy) and ramp to p=6 (strict) after 200 epochs.

**Is Forall always pMeanError and Exists always pMean?**  
No — it is a convention, not a rule. `Wrapper_Quantifier` accepts any aggregation function. You could use `Aggreg_Min` for a strict classical Forall, or `Aggreg_Mean` for a soft one. The semantics argument (`"forall"` / `"exists"`) only controls the behaviour when a quantified set is empty (empty Forall → 1, empty Exists → 0). The actual aggregation is entirely determined by the function you pass in.

In [ ]:
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(p=2), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(p=6),      semantics="exists")

# The formula_aggregator folds a LIST of formula scalars into a single sat score
# (used to combine multiple axioms into one loss value)
formula_aggregator = ltn.Wrapper_Formula_Aggregator(ltn.fuzzy_ops.Aggreg_pMeanError())

### 4a. Seeing the fold in action

Let's manually trace what happens when we evaluate `∀p Smokes(p)` before training.

In [ ]:
# Build a Variable = a batch of all person embeddings stacked together
p = ltn.Variable.from_constants("p", list(g.values()))
print("Variable tensor shape (n_people, embedding_size):", p.tensor.shape)

# Evaluate Smokes on all people at once → tensor of shape (n_people,)
smokes_values = Smokes(p)
print("Smokes(p) shape:", smokes_values.tensor.shape)
print("Smokes(p) values:", smokes_values.tensor.numpy().round(3))

# Forall FOLDS that (n_people,) tensor → single scalar
forall_smokes = Forall(p, Smokes(p))
print("\nForall(p, Smokes(p)) shape:", forall_smokes.tensor.shape)
print("Forall(p, Smokes(p)) value:", forall_smokes.tensor.numpy().round(4))
print("(should be low — not everyone smokes in the data)")

# Manual check: reproduce the fold by hand
vals = smokes_values.tensor.numpy()
p_val = 2
manual = 1. - (np.mean((1. - vals)**p_val))**(1./p_val)
print(f"\nManual pMeanError(p={p_val}): {manual:.4f}  ← should match Forall above")

## 5. The knowledge base (axioms)

We encode the domain rules as first-order logic formulas.  
Each axiom returns a satisfaction value in [0,1].  
The overall **sat_level** is the fold of all axiom values — the single number we maximise.

The axioms are:
1. Observed friendship pairs → `Friends(x,y) ≈ 1`
2. Non-friend pairs → `Friends(x,y) ≈ 0`
3. Observed smokers → `Smokes(x) ≈ 1`
4. Observed non-smokers → `Smokes(x) ≈ 0`
5. Observed cancer cases → `Cancer(x) ≈ 1`
6. Observed non-cancer (in g1) → `Cancer(x) ≈ 0`
7. Friendship is anti-reflexive: `∀p ¬Friends(p,p)`
8. Friendship is symmetric: `∀p,q Friends(p,q) ⟹ Friends(q,p)`
9. Everyone has a friend: `∀p ∃q Friends(p,q)`
10. Smoking spreads: `∀p,q Friends(p,q) ∧ Smokes(p) ⟹ Smokes(q)`
11. Smoking causes cancer: `∀p Smokes(p) ⟹ Cancer(p)`
12. Not smoking ⟹ no cancer: `∀p ¬Smokes(p) ⟹ ¬Cancer(p)`

In [ ]:
@tf.function
def axioms(p_exists):
    # Re-build variables each step because the constant embeddings are trainable
    p = ltn.Variable.from_constants("p", list(g.values()))
    q = ltn.Variable.from_constants("q", list(g.values()))

    ax = []

    # --- Observed facts (fold a list of ground atoms) ---
    ax.append(formula_aggregator([Friends([g[x], g[y]]) for (x,y) in friends]))
    ax.append(formula_aggregator(
        [Not(Friends([g[x], g[y]])) for x in g1 for y in g1 if (x,y) not in friends and x < y] +
        [Not(Friends([g[x], g[y]])) for x in g2 for y in g2 if (x,y) not in friends and x < y]
    ))
    ax.append(formula_aggregator([Smokes(g[x]) for x in smokes]))
    ax.append(formula_aggregator([Not(Smokes(g[x])) for x in g if x not in smokes]))
    ax.append(formula_aggregator([Cancer(g[x]) for x in cancer]))
    ax.append(formula_aggregator([Not(Cancer(g[x])) for x in g1 if x not in cancer]))

    # --- General rules (quantifiers = fold over all individuals) ---
    ax.append(Forall(p, Not(Friends([p, p])), p=5))                          # anti-reflexive
    ax.append(Forall((p,q), Implies(Friends([p,q]), Friends([q,p])), p=5))   # symmetric
    ax.append(Forall(p, Exists(q, Friends([p,q]), p=p_exists)))              # everyone has a friend
    ax.append(Forall((p,q), Implies(And(Friends([p,q]), Smokes(p)), Smokes(q))))  # smoking spreads
    ax.append(Forall(p, Implies(Smokes(p), Cancer(p))))                      # smoking → cancer
    ax.append(Forall(p, Implies(Not(Smokes(p)), Not(Cancer(p)))))            # ¬smoking → ¬cancer

    # --- Final fold: combine all axiom sat values into one scalar ---
    sat_level = formula_aggregator(ax).tensor
    return sat_level

print("Initial sat level: %.4f" % axioms(p_exists=tf.constant(6.)).numpy())

## 6. Training loop

**Loss = 1 − sat_level**  
We minimise the loss ↔ maximise how well the model satisfies all axioms simultaneously.

Trainable parameters:
- Weights of `Smokes`, `Friends`, `Cancer` MLPs
- The embedding vector of each person (trainable constants)

Note the curriculum on `p_exists`: start with p=1 (soft `Exists`) so the model easily satisfies
"everyone has a friend", then increase to p=6 (strict) after epoch 200.

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

trainable_variables = (
    Smokes.trainable_variables +
    Friends.trainable_variables +
    Cancer.trainable_variables +
    ltn.as_tensors(list(g.values()))   # the person embedding vectors
)

@tf.function
def train_step(p_exists):
    with tf.GradientTape() as tape:
        sat   = axioms(p_exists)
        loss  = 1. - sat
    grads = tape.gradient(loss, trainable_variables)
    optimizer.apply_gradients(zip(grads, trainable_variables))
    return sat

In [ ]:
EPOCHS = 1000

for epoch in range(EPOCHS):
    p_exists = tf.constant(1.) if epoch < 200 else tf.constant(6.)
    sat = train_step(p_exists)
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}  sat={sat.numpy():.4f}")

## 7. Querying the trained model

After training we can evaluate formulas that were **not** in the knowledge base.  
The model has to *infer* answers from what it learned.

In [ ]:
p = ltn.Variable.from_constants("p", list(g.values()))
q = ltn.Variable.from_constants("q", list(g.values()))

# φ1: Does having cancer imply smoking? (not stated — should be inferred)
phi1 = Forall(p, Implies(Cancer(p), Smokes(p)), p=5)
print(f"φ1  ∀p Cancer(p)→Smokes(p):          {phi1.tensor.numpy():.4f}")

# φ2: Does cancer co-occur with friendship? (not stated)
phi2 = Forall((p,q), Implies(Or(Cancer(p), Cancer(q)), Friends([p,q])), p=5)
print(f"φ2  ∀p,q Cancer(p)∨Cancer(q)→Friends: {phi2.tensor.numpy():.4f}")

# Individual predictions for g2 (labels were unknown during training)
print("\nPredictions for g2 (semi-supervised):")
for name in 'ijklmn':
    s = Smokes(g[name]).tensor.numpy()
    c = Cancer(g[name]).tensor.numpy()
    print(f"  {name}: Smokes={s:.3f}  Cancer={c:.3f}")

## Summary

| Concept | LTN implementation |
|---------|-------------------|
| Individual (constant) | trainable embedding vector |
| Predicate | MLP with sigmoid output → [0,1] |
| ¬, ∧, ∨, ⟹ | element-wise fuzzy ops on tensors |
| **∀x P(x)** | **fold (reduce) over axis with pMeanError** |
| **∃x P(x)** | **fold (reduce) over axis with pMean** |
| Knowledge base | list of axiom scalars → fold into sat_level |
| Training | minimise `1 - sat_level` with Adam |

**Next:** `02_grounding_and_variables.ipynb` — deep dive into how tensor shapes change as variables are combined in formulas.